# CSV Basics — 01: Reading CSV Files

## What you will learn
Reading CSV is the most common Spark task for data engineers.
Despite its simplicity, CSV has many hidden options that dramatically
affect correctness and performance.

In this notebook you will cover:
1. Basic `spark.read.csv()` with default options
2. All critical read options explained (header, sep, encoding, multiline)
3. Reading a single file vs directory vs glob pattern
4. Schema inference vs explicit schema — why explicit is always better
5. Handling null values and custom null markers


In [1]:
import os, time, pathlib, shutil
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data/csv_basics'
pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName('csv-basics')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version} | DATA_DIR: {DATA_DIR}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/10 13:15:59 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.0.2 | DATA_DIR: /workspace/data/csv_basics


## Step 1 — Create Sample CSV Files

We create several CSV files with different characteristics to test all read options.


In [2]:
import pathlib

# Basic CSV with header
basic_csv = """order_id,customer_id,product,quantity,price,status
1,101,Laptop,1,999.99,confirmed
2,102,Phone,2,599.99,shipped
3,103,Tablet,,349.99,pending
4,104,Headphones,1,,delivered
5,105,Monitor,3,449.99,confirmed"""

# CSV with different separator
pipe_csv = """order_id|customer_id|product|amount
1|101|Laptop|999.99
2|102|Phone|599.99
3|103|Tablet|349.99"""

# CSV with quotes and commas inside values
quoted_csv = '''order_id,product,description,price
1,Laptop,"High-end laptop, 16GB RAM",999.99
2,Phone,"Smartphone, 5G capable",599.99
3,Book,"Introduction to Spark, 3rd edition",49.99'''

# CSV without header
no_header_csv = """1,101,Laptop,999.99
2,102,Phone,599.99
3,103,Tablet,349.99"""

# Multiline CSV (value spans multiple lines)
multiline_csv = '''id,name,notes
1,Alice,"She works in
engineering dept"
2,Bob,"Standard
employee"'''

for name, content in [
    ("basic.csv",       basic_csv),
    ("pipe_sep.csv",    pipe_csv),
    ("quoted.csv",      quoted_csv),
    ("no_header.csv",   no_header_csv),
    ("multiline.csv",   multiline_csv),
]:
    pathlib.Path(f"{DATA_DIR}/{name}").write_text(content)
    print(f"Created: {name}")

Created: basic.csv
Created: pipe_sep.csv
Created: quoted.csv
Created: no_header.csv
Created: multiline.csv


## Step 2 — Basic Read with Default Options

`spark.read.csv()` with no options uses sensible defaults:
- separator: `,`
- header: `false` (first row treated as data!)
- inferSchema: `false` (everything is a string)
- nullValue: `""` (empty string = null)


In [3]:
# Default read — no options
df_default = spark.read.csv(f"{DATA_DIR}/basic.csv")
print("Default read (no options) — notice _c0, _c1 column names:")
df_default.printSchema()
df_default.show()

print("\nProblem 1: header row is treated as data (row 0 = column names as values)")
print("Problem 2: all columns are strings, even numeric ones")
print("Problem 3: columns named _c0, _c1, _c2... not meaningful")

Default read (no options) — notice _c0, _c1 column names:
root
 |-- _c0: string (nullable = true)
 |-- _c1: string (nullable = true)
 |-- _c2: string (nullable = true)
 |-- _c3: string (nullable = true)
 |-- _c4: string (nullable = true)
 |-- _c5: string (nullable = true)



[Stage 1:>                                                          (0 + 1) / 1]

+--------+-----------+----------+--------+------+---------+
|     _c0|        _c1|       _c2|     _c3|   _c4|      _c5|
+--------+-----------+----------+--------+------+---------+
|order_id|customer_id|   product|quantity| price|   status|
|       1|        101|    Laptop|       1|999.99|confirmed|
|       2|        102|     Phone|       2|599.99|  shipped|
|       3|        103|    Tablet|    NULL|349.99|  pending|
|       4|        104|Headphones|       1|  NULL|delivered|
|       5|        105|   Monitor|       3|449.99|confirmed|
+--------+-----------+----------+--------+------+---------+


Problem 1: header row is treated as data (row 0 = column names as values)
Problem 2: all columns are strings, even numeric ones
Problem 3: columns named _c0, _c1, _c2... not meaningful


## Step 3 — Essential Options: header, sep, inferSchema


In [4]:
# With header=True
df_with_header = spark.read.csv(f"{DATA_DIR}/basic.csv", header=True)
print("With header=True — column names from first row:")
df_with_header.printSchema()
df_with_header.show()

print("\nAll columns are still StringType — inferSchema=False by default")

With header=True — column names from first row:
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- price: string (nullable = true)
 |-- status: string (nullable = true)

+--------+-----------+----------+--------+------+---------+
|order_id|customer_id|   product|quantity| price|   status|
+--------+-----------+----------+--------+------+---------+
|       1|        101|    Laptop|       1|999.99|confirmed|
|       2|        102|     Phone|       2|599.99|  shipped|
|       3|        103|    Tablet|    NULL|349.99|  pending|
|       4|        104|Headphones|       1|  NULL|delivered|
|       5|        105|   Monitor|       3|449.99|confirmed|
+--------+-----------+----------+--------+------+---------+


All columns are still StringType — inferSchema=False by default


In [5]:
# With header=True and inferSchema=True
df_inferred = spark.read.csv(f"{DATA_DIR}/basic.csv",
                              header=True, inferSchema=True)
print("With inferSchema=True — Spark guesses data types:")
df_inferred.printSchema()
df_inferred.show()
print("\nNote: Spark scans the ENTIRE file twice for inferSchema — expensive on large files!")
print("Recommendation: always define explicit schema for production pipelines.")

[Stage 5:>                                                          (0 + 1) / 1]

With inferSchema=True — Spark guesses data types:
root
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- product: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- price: double (nullable = true)
 |-- status: string (nullable = true)

+--------+-----------+----------+--------+------+---------+
|order_id|customer_id|   product|quantity| price|   status|
+--------+-----------+----------+--------+------+---------+
|       1|        101|    Laptop|       1|999.99|confirmed|
|       2|        102|     Phone|       2|599.99|  shipped|
|       3|        103|    Tablet|    NULL|349.99|  pending|
|       4|        104|Headphones|       1|  NULL|delivered|
|       5|        105|   Monitor|       3|449.99|confirmed|
+--------+-----------+----------+--------+------+---------+


Note: Spark scans the ENTIRE file twice for inferSchema — expensive on large files!
Recommendation: always define explicit schema for production pipelines.


In [6]:
# Different separator
df_pipe = spark.read.csv(f"{DATA_DIR}/pipe_sep.csv",
                          header=True, sep="|", inferSchema=True)
print("Pipe-separated CSV (sep='|'):")
df_pipe.show()

# Common separators:
print("\nCommon sep options:")
print("  sep=','   comma (default)")
print("  sep='|'   pipe")
print("  sep='\t'  tab (TSV)")
print("  sep=';'   semicolon (European CSV)")

Pipe-separated CSV (sep='|'):
+--------+-----------+-------+------+
|order_id|customer_id|product|amount|
+--------+-----------+-------+------+
|       1|        101| Laptop|999.99|
|       2|        102|  Phone|599.99|
|       3|        103| Tablet|349.99|
+--------+-----------+-------+------+


Common sep options:
  sep=','   comma (default)
  sep='|'   pipe
  sep='	'  tab (TSV)
  sep=';'   semicolon (European CSV)


## Step 4 — Explicit Schema: The Right Way

Always define schema explicitly in production. Benefits:
- No double scan of the file
- Guaranteed types (no surprises)
- Better error handling for bad data
- Self-documenting code


In [7]:
# Explicit schema — the production-grade approach
order_schema = StructType([
    StructField("order_id",    IntegerType(),  nullable=False),
    StructField("customer_id", IntegerType(),  nullable=False),
    StructField("product",     StringType(),   nullable=False),
    StructField("quantity",    IntegerType(),  nullable=True),   # nullable: can be missing
    StructField("price",       DoubleType(),   nullable=True),
    StructField("status",      StringType(),   nullable=True),
])

df_explicit = spark.read.csv(f"{DATA_DIR}/basic.csv",
                              header=True, schema=order_schema)
print("Explicit schema — types are guaranteed:")
df_explicit.printSchema()
df_explicit.show()
print("Row 3 has null quantity, Row 4 has null price — handled correctly")

Explicit schema — types are guaranteed:
root
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- product: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- price: double (nullable = true)
 |-- status: string (nullable = true)

+--------+-----------+----------+--------+------+---------+
|order_id|customer_id|   product|quantity| price|   status|
+--------+-----------+----------+--------+------+---------+
|       1|        101|    Laptop|       1|999.99|confirmed|
|       2|        102|     Phone|       2|599.99|  shipped|
|       3|        103|    Tablet|    NULL|349.99|  pending|
|       4|        104|Headphones|       1|  NULL|delivered|
|       5|        105|   Monitor|       3|449.99|confirmed|
+--------+-----------+----------+--------+------+---------+

Row 3 has null quantity, Row 4 has null price — handled correctly


In [8]:
# Alternative: DDL schema string (more concise)
ddl_schema = "order_id INT NOT NULL, customer_id INT, product STRING, quantity INT, price DOUBLE, status STRING"
df_ddl = spark.read.csv(f"{DATA_DIR}/basic.csv", header=True, schema=ddl_schema)
print("DDL schema string (shorter syntax):")
df_ddl.printSchema()
df_ddl.show(3)

DDL schema string (shorter syntax):
root
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- product: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- price: double (nullable = true)
 |-- status: string (nullable = true)

+--------+-----------+-------+--------+------+---------+
|order_id|customer_id|product|quantity| price|   status|
+--------+-----------+-------+--------+------+---------+
|       1|        101| Laptop|       1|999.99|confirmed|
|       2|        102|  Phone|       2|599.99|  shipped|
|       3|        103| Tablet|    NULL|349.99|  pending|
+--------+-----------+-------+--------+------+---------+
only showing top 3 rows


## Step 5 — Reading Multiple Files: Directory, Glob, List


In [9]:
# Create multiple CSV files in a directory
multi_dir = f"{DATA_DIR}/multi"
pathlib.Path(multi_dir).mkdir(exist_ok=True)

for month in range(1, 4):
    content = f"order_id,amount,month\n"
    for i in range(3):
        content += f"{month*10+i},{(month*10+i)*50.0},{month}\n"
    pathlib.Path(f"{multi_dir}/orders_2024_0{month}.csv").write_text(content)

print("Created 3 CSV files in multi/ directory")

schema_m = "order_id INT, amount DOUBLE, month INT"

# 1. Read entire directory
df_dir = spark.read.csv(f"{multi_dir}/", header=True, schema=schema_m)
print(f"\n1. Read directory — {df_dir.count()} rows from all files:")
df_dir.show()

# 2. Months 1 and 2 only — Python-resolved list (glob strings unreliable on local FS)
q1_files = [f"{multi_dir}/orders_2024_01.csv", f"{multi_dir}/orders_2024_02.csv"]
df_list_12 = spark.read.csv(q1_files, header=True, schema=schema_m)
print(f"2. Months 1-2 (explicit list) — {df_list_12.count()} rows:")
df_list_12.show()

# 3. Read specific files as list
df_list = spark.read.csv(
    [f"{multi_dir}/orders_2024_01.csv", f"{multi_dir}/orders_2024_03.csv"],
    header=True, schema=schema_m
)
print(f"3. List of files (months 1 and 3) — {df_list.count()} rows:")
df_list.show()

# 4. Use Python glob to resolve patterns, then pass list to Spark
all_files = sorted(glib.glob(f"{multi_dir}/*.csv"))
df_all = spark.read.csv(all_files, header=True, schema=schema_m)
print(f"4. Python glob resolved {len(all_files)} files — {df_all.count()} rows:")
df_all.show()

print()
print("Note: glob patterns like '*.csv' or '[12].csv' do NOT work reliably")
print("on local filesystem in Spark 4.x. Use one of:")
print("  spark.read.csv('/path/dir/')              # whole directory")
print("  spark.read.csv(['/path/a.csv', '/path/b.csv'])  # explicit list")
print("  spark.read.csv(glob.glob('/path/*.csv'))   # Python-resolved glob")

Created 3 CSV files in multi/ directory

1. Read directory — 9 rows from all files:
+--------+------+-----+
|order_id|amount|month|
+--------+------+-----+
|      20|1000.0|    2|
|      21|1050.0|    2|
|      22|1100.0|    2|
|      30|1500.0|    3|
|      31|1550.0|    3|
|      32|1600.0|    3|
|      10| 500.0|    1|
|      11| 550.0|    1|
|      12| 600.0|    1|
+--------+------+-----+

2. Months 1-2 (explicit list) — 6 rows:
+--------+------+-----+
|order_id|amount|month|
+--------+------+-----+
|      20|1000.0|    2|
|      21|1050.0|    2|
|      22|1100.0|    2|
|      10| 500.0|    1|
|      11| 550.0|    1|
|      12| 600.0|    1|
+--------+------+-----+

3. List of files (months 1 and 3) — 6 rows:
+--------+------+-----+
|order_id|amount|month|
+--------+------+-----+
|      30|1500.0|    3|
|      31|1550.0|    3|
|      32|1600.0|    3|
|      10| 500.0|    1|
|      11| 550.0|    1|
|      12| 600.0|    1|
+--------+------+-----+



NameError: name 'glib' is not defined

## Step 6 — Null Value Handling

CSV has no native concept of null — empty strings or special markers
like `NA`, `N/A`, `NULL`, `\N` need explicit configuration.


In [ ]:
# CSV with various null representations
null_csv = """id,value,category,score
1,100,A,9.5
2,,B,
3,300,N/A,8.0
4,400,NULL,7.5
5,500,\N,"""

pathlib.Path(f"{DATA_DIR}/nulls.csv").write_text(null_csv)

schema_n = "id INT, value INT, category STRING, score DOUBLE"

# Default: only empty string = null
df_default_null = spark.read.csv(f"{DATA_DIR}/nulls.csv",
                                  header=True, schema=schema_n)
print("Default null handling (only empty string = null):")
df_default_null.show()
df_default_null.select([F.count(F.when(F.col(c).isNull(), c)).alias(c)
                        for c in df_default_null.columns]).show()

# With multiple null values
df_nulls = spark.read.csv(f"{DATA_DIR}/nulls.csv",
    header=True, schema=schema_n,
    nullValue="N/A",    # treat N/A as null
    nanValue="NULL"     # treat NULL string as NaN (numeric)
)
print("\nWith nullValue='N/A':")
df_nulls.show()

## Step 7 — Quoted Values and Multiline

CSV values can contain commas or newlines if wrapped in quotes.
Spark handles this with `quote`, `escape`, and `multiLine` options.


In [ ]:
# Quoted values with commas inside
df_quoted = spark.read.csv(f"{DATA_DIR}/quoted.csv",
    header=True, inferSchema=True,
    quote='"\'',   # default quote char is double-quote
    escape='"\'')  # default escape is same as quote (doubled)
print("Quoted CSV (commas inside values):")
df_quoted.show(truncate=False)

# Multiline values
df_multi = spark.read.csv(f"{DATA_DIR}/multiline.csv",
    header=True, multiLine=True, quote='"\'')
print("\nMultiline CSV:")
df_multi.show(truncate=False)
print("\nNote: multiLine=True disables parallel reads — file must fit in one partition")
print("Avoid multiLine for large files. Consider pre-processing to single-line format.")

## Summary: CSV Read Options Cheat Sheet

```python
spark.read.csv(
    path,
    header       = True,          # first row = column names
    sep          = ',',           # or '|', '\t', ';'
    schema       = your_schema,   # ALWAYS explicit in production
    inferSchema  = False,         # never in production (2x scan)
    nullValue    = 'N/A',         # custom null marker
    nanValue     = 'NaN',
    quote        = '"',           # quote character
    escape       = '"',           # escape character
    multiLine    = False,         # True only for small files
    encoding     = 'UTF-8',       # or 'latin1', 'windows-1252'
    dateFormat   = 'yyyy-MM-dd',
    timestampFormat = 'yyyy-MM-dd HH:mm:ss',
    mode         = 'PERMISSIVE',  # or DROPMALFORMED, FAILFAST
    ignoreLeadingWhiteSpace  = True,
    ignoreTrailingWhiteSpace = True,
)
```

**Rule of thumb:** always set `header=True`, `schema=your_schema`, and the appropriate
`nullValue` for your data source. Never use `inferSchema=True` in production.
